<a href="https://colab.research.google.com/github/alexa21342/CSCI-167/blob/CSCI-167-Assignments/12_1_Self_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Alex Cortez CSCI 167 11/10/2025

# **Notebook 12.1: Self Attention**

This notebook builds a self-attention mechanism from scratch, as discussed in section 12.2 of the book.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [71]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [72]:
# Set seed so we get the same random numbers
np.random.seed(3)
# Number of inputs
N = 3
# Number of dimensions of each input
D = 4
# Create an empty list
all_x = []
# Create elements x_n and append to list
for n in range(N):
  all_x.append(np.random.normal(size=(D,1)))
# Print out the list
print(all_x)


[array([[ 1.78862847],
       [ 0.43650985],
       [ 0.09649747],
       [-1.8634927 ]]), array([[-0.2773882 ],
       [-0.35475898],
       [-0.08274148],
       [-0.62700068]]), array([[-0.04381817],
       [-0.47721803],
       [-1.31386475],
       [ 0.88462238]])]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [73]:
# Set seed so we get the same random numbers
np.random.seed(0)

# Choose random values for the parameters
omega_q = np.random.normal(size=(D,D))
omega_k = np.random.normal(size=(D,D))
omega_v = np.random.normal(size=(D,D))
beta_q = np.random.normal(size=(D,1))
beta_k = np.random.normal(size=(D,1))
beta_v = np.random.normal(size=(D,1))

Now let's compute the queries, keys, and values for each input

In [74]:
# Make three lists to store queries, keys, and values
all_queries = []
all_keys = []
all_values = []
# For every input
for x in all_x:
  # TODO -- compute the keys, queries and values.
  # Replace these three lines
  # query = np.ones_like(x)
  # key = np.ones_like(x)
  # value = np.ones_like(x)

  # Example linear transformations for queries, keys, and values
  # (using random or predefined weight matrices for demonstration)
  # If Wq, Wk, Wv are defined elsewhere, use them here instead.
  Wq = np.random.randn(x.shape[0], x.shape[0])
  Wk = np.random.randn(x.shape[0], x.shape[0])
  Wv = np.random.randn(x.shape[0], x.shape[0])

  query = np.dot(Wq, x)
  key = np.dot(Wk, x)
  value = np.dot(Wv, x)

  all_queries.append(query)
  all_keys.append(key)
  all_values.append(value)


We'll need a softmax function (equation 12.5) -- here, it will take a list of arbitrary numbers and return a list where the elements are non-negative and sum to one


In [75]:
def softmax(items_in):

  # TODO Compute the elements of items_out
  # Replace this line
  # items_out = items_in.copy()

  # Subtract max for numerical stability
  exp_items = np.exp(items_in - np.max(items_in))
  items_out = exp_items / np.sum(exp_items)

  return items_out


Now compute the self attention values:

In [76]:
# Create empty list for output
all_x_prime = []

# For each output
for n in range(N):
    # Create list for dot products of query N with all keys
    all_km_qn = []
    # Compute the dot products
    for key in all_keys:
        # TODO -- compute the appropriate dot product
        # Replace this line
        dot_product = np.dot(all_queries[n].T, key)  # scalar similarity

        # Store dot product
        all_km_qn.append(dot_product)

    # Compute attention weights
    attention = softmax(all_km_qn)
    # Print result (should be positive and sum to one)
    print("Attentions for output ", n)
    print(attention)

    # TODO: Compute a weighted sum of all of the values according to the attention
    # (equation 12.3)
    # Replace this line
    x_prime = np.zeros((D, 1))
    for m in range(len(all_values)):
        x_prime += attention[m] * all_values[m]

    all_x_prime.append(x_prime)

# Print out true values to check you have it correct
print("x_prime_0_calculated:", all_x_prime[0].transpose())
print("x_prime_0_true: [[ 0.94744244 -0.24348429 -0.91310441 -0.44522983]]")
print("x_prime_1_calculated:", all_x_prime[1].transpose())
print("x_prime_1_true: [[ 1.64201168 -0.08470004  4.02764044  2.18690791]]")
print("x_prime_2_calculated:", all_x_prime[2].transpose())
print("x_prime_2_true: [[ 1.61949281 -0.06641533  3.96863308  2.15858316]]")


Attentions for output  0
[[[9.26314455e-15]]

 [[9.24499305e-05]]

 [[9.99907550e-01]]]
Attentions for output  1
[[[3.18842919e-01]]

 [[6.80860957e-01]]

 [[2.96123911e-04]]]
Attentions for output  2
[[[0.05471672]]

 [[0.50180356]]

 [[0.44347973]]]
x_prime_0_calculated: [[ 1.23080216 -0.28023587 -2.08654341 -0.73457675]]
x_prime_0_true: [[ 0.94744244 -0.24348429 -0.91310441 -0.44522983]]
x_prime_1_calculated: [[-0.25875831 -1.08413141 -0.09688365  0.6617135 ]]
x_prime_1_true: [[ 1.64201168 -0.08470004  4.02764044  2.18690791]]
x_prime_2_calculated: [[ 0.44207241 -0.9342641  -1.14974524  0.14342933]]
x_prime_2_true: [[ 1.61949281 -0.06641533  3.96863308  2.15858316]]


Now let's compute the same thing, but using matrix calculations.  We'll store the $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ in the columns of a $D\times N$ matrix, using equations 12.6 and 12.7/8.

Note:  The book uses column vectors (for compatibility with the rest of the text), but in the wider literature it is more normal to store the inputs in the rows of a matrix;  in this case, the computation is the same, but all the matrices are transposed and the operations proceed in the reverse order.

In [77]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax

In [78]:
# Now let's compute self attention in matrix form
def self_attention(X, omega_v, omega_q, omega_k, beta_v, beta_q, beta_k):

  # TODO -- Write this function
  # 1. Compute queries, keys, and values
  # 2. Compute dot products
  # 3. Apply softmax to calculate attentions
  # 4. Weight values by attentions
  # Replace this line

  # 1. Compute queries, keys, and values (linear transformations)
  Q = X @ omega_q + beta_q.T      # shape: (N, D)
  K = X @ omega_k + beta_k.T      # shape: (N, D)
  V = X @ omega_v + beta_v.T      # shape: (N, D)

  # 2. Compute dot products between queries and keys
  # Each element [i, j] corresponds to q_i^T k_j
  dot_products = Q @ K.T        # shape: (N, N)

  # 3. Apply softmax along the key dimension (axis=1)
  # for numerical stability, subtract max per row
  exp_dot = np.exp(dot_products - np.max(dot_products, axis=1, keepdims=True))
  attention = exp_dot / np.sum(exp_dot, axis=1, keepdims=True)  # shape: (N, N)

  # 4. Weight values by attentions
  X_prime = attention @ V       # shape: (N, D)

  return X_prime

In [79]:
# Copy data into matrix
X = np.zeros((D, N))
X[:,0] = np.squeeze(all_x[0])
X[:,1] = np.squeeze(all_x[1])
X[:,2] = np.squeeze(all_x[2])

# Run the self attention mechanism
X_prime = self_attention(X.T,omega_v, omega_q, omega_k, beta_v, beta_q, beta_k)

# Print out the results
print(X_prime)

[[ 0.43250036  1.25670068  0.6858925  -2.0111436 ]
 [ 0.00828235 -1.08920945  0.68373683 -1.37055217]
 [-0.11939362 -1.92678975  0.73672173 -1.46037153]]


If you did this correctly, the values should be the same as above.

TODO:  

Print out the attention matrix
You will see that the values are quite extreme (one is very close to one and the others are very close to zero.  Now we'll fix this problem by using scaled dot-product attention.

In [80]:
# Now let's compute self attention in matrix form
def scaled_dot_product_self_attention(X, omega_v, omega_q, omega_k, beta_v, beta_q, beta_k):

  # TODO -- Write this function
  # 1. Compute queries, keys, and values
  # 2. Compute dot products
  # 3. Scale the dot products as in equation 12.9
  # 4. Apply softmax to calculate attentions
  # 5. Weight values by attentions
  # Replace this line
  # X_prime = np.zeros_like(X);

  # 1. Compute queries, keys, and values
  Q = X @ omega_q + beta_q.T
  K = X @ omega_k + beta_k.T
  V = X @ omega_v + beta_v.T

  # 2. Compute dot products
  dot_products = Q @ K.T  # shape (N, N)

  # 3. Scale the dot products as in equation 12.9
  d_k = K.shape[1]        # dimensionality of key vectors
  scaled_dot = dot_products / np.sqrt(d_k)

  # 4. Apply softmax to calculate attentions
  exp_dot = np.exp(scaled_dot - np.max(scaled_dot, axis=1, keepdims=True))
  attention = exp_dot / np.sum(exp_dot, axis=1, keepdims=True)

  # 5. Weight values by attentions
  X_prime = attention @ V

  return X_prime

In [81]:
# Run the self attention mechanism
X_prime = scaled_dot_product_self_attention(X.T, omega_v, omega_q, omega_k, beta_v, beta_q, beta_k).T

# Print out the results
print(X_prime)


[[ 0.40797669  0.08863953 -0.04331903]
 [ 1.18303771 -0.60075218 -1.43801028]
 [ 0.66050715  0.66616966  0.70934494]
 [-1.84100293 -1.39717579 -1.42895216]]


TODO -- Investigate whether the self-attention mechanism is covariant with respect to permutation.
If it is, when we permute the columns of the input matrix $\mathbf{X}$, the columns of the output matrix $\mathbf{X}'$ will also be permuted.
